# Assignment 2:
### Hierarchical Clustering
Date: September 24, 2026

Report produced by:  
COMP 3705 Group 3

In [ ]:
import pandas as pd

df_cc_general = pd.read_csv("CC GENERAL.csv")
df_cc_general

In [ ]:
import numpy as np
from matplotlib import pyplot as plt
from sklearn.preprocessing import StandardScaler as scaler
from sklearn.cluster import AgglomerativeClustering as ac
from sklearn.metrics import silhouette_score as s_score, silhouette_samples as s_samples

df_cc_cleaned = df_cc_general.drop(columns="CUST_ID")

def scale_df(df):
    return scaler().fit_transform(df)

def agglomerative_clustering(x_val, min_clusters=2, max_clusters=3, distance_metric=["euclidean"], linkage=["ward"], max_linkage_distance=[None]):
    # Create containers for results
    iteration = 0
    ac_metrics = {}

    # Sets min_clusters to 1 if value passed is lower
    if min_clusters < 1:
        min_clusters = 1    

    
    for num in range(min_clusters, max_clusters+1):
        for distance in distance_metric:
            for link in linkage:
                if link == "ward" and distance != "euclidean":
                    continue
                for threshold in max_linkage_distance:
                    if threshold != None:
                        model = ac(n_clusters=None, affinity=distance, linkage=link, distance_threshold=threshold, compute_full_tree=True).fit(x_val)
                    else:
                        model = ac(n_clusters=num, affinity=distance, linkage=link).fit(x_val)

                    iteration += 1
                    labels = model.labels_
                    parameters = {"n_clusters": num,
                                "distance_metric": distance,
                                "linkage": link,
                                "distance_threshold": threshold,
                                }
                    metrics = {"Silhouette Score": s_score(x_val,labels,metric=distance),
                            "Silhouette Coefficients": s_samples(x_val,labels,metric=distance),
                            "Labels": model.labels_,
                            "Number of Clusters": model.n_clusters_,
                            "Merge History": model.children_,
                            "Merge Distances": model.distances_,
                            }
                    ac_metrics[f"Iteration {iteration} Parameters"] = parameters
                    ac_metrics[f"Iteration {iteration} Metrics"] = metrics

    return ac_metrics

df_scaled = scale_df(df_cc_cleaned)
ac_metrics = agglomerative_clustering(df_scaled, 3, 5, ["euclidean", "manhattan", "cosine"],["ward", "average"])

silhouette_avg = []
for i in range(1,len(ac_metrics)/2+1):
    silhouette_avg.append(ac_metrics[f"Iteration {i} Metrics"]["Silhouette Score"])
print(silhouette_avg)

In [ ]:
import numpy as np
from scipy.cluster.hierarchy import linkage, fcluster
from sklearn.metrics import silhouette_score

def scipy_modeling(features_scaled):
    # Create the hierarchical tree
    Z = linkage(features_scaled, method="ward")
    
    # Create 10 different model settings
    models = [
        (2, "maxclust", 2),
        (3, "maxclust", 2),
        (4, "maxclust", 2),
        (5, "maxclust", 2),
        (6, "maxclust", 2),
        (20, "distance", 2),
        (22, "distance", 2),
        (25, "distance", 2),
        (1.0, "inconsistent", 2),
        (1.0, "inconsistent", 3)
    ]
    
    # Empty list for storing results
    results = []
    
    # Run all 10 models
    for i, (t, criterion, depth) in enumerate(models, start=1):
        clusters = fcluster(
            Z,
            t=t,
            criterion=criterion,
            depth=depth
        )
        number_of_clusters = len(np.unique(clusters))
        if number_of_clusters > 1:
            score = silhouette_score(features_scaled, clusters)
        else:
            score = None
        results.append({
            "Model": i,
            "t": t,
            "Criterion": criterion,
            "Depth": depth,
            "Number of Clusters": number_of_clusters,
            "Silhouette Score": score
        })
    return results

scipy_results = scipy_modeling(df_scaled)

## Acknowledgment of Contributions
### Group 3 Members:  
Sebastian Levy  
Idongesit Orok  
Miracle Okojie  

### Task Distribution:  


We, Group 3, acknowledge that we understand all parts of this assignment, including those which we did not personally carry out.